##Configuration

In [0]:
import dlt
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

CATALOG = spark.conf.get("pipeline.catalog", "vstone_catalog")
BRONZE = spark.conf.get("pipeline.bronze_schema", "bronze")

SILVER_PROPS = {
    "quality": "silver",
    "delta.enableChangeDataFeed": "true",
    "pipelines.reset.allowed": "true",
}
QUARANTINE_PROPS = {
    "quality": "quarantine",
    "delta.enableChangeDataFeed": "true",
    "pipelines.reset.allowed": "true",
}


## Pandas UDF: value-level cleaning 

In [0]:
@F.pandas_udf(StringType())
def clean_text(s: pd.Series) -> pd.Series:
    """Strip whitespace only, preserve case — same UDF as 09_silver_streets.py.
    Duplicated rather than imported: DLT pipeline notebooks don't reliably
    resolve cross-file module imports, and this is 3 lines."""
    return s.astype(str).str.strip()

## Bronze Stream

In [0]:
def _bronze_stream(table):
    return spark.readStream.format("delta").table(f"{CATALOG}.{BRONZE}.{table}")

## Deduplicate

In [0]:
def deduplicate(df, partition_cols: list):
    return df.dropDuplicates(partition_cols)

## _cars_silver_ Transformation

In [0]:
# Grain: (location, date) per the Day 1 assumption doc — id (0-998, cycling)
# is not a key. UNLIKE streets, this grain has NOT been empirically verified
# duplicate-free (we never ran that check against the real 24.7M-row file
# the way we did for streets). dropDuplicates is applied on that assumed
# grain regardless — test_silver_day4.py checks the actual duplicate count
# post-hoc so this isn't just asserted and forgotten.

def _transform_cars(df):
    return (df.select(
        F.expr("try_cast(location as int)").alias("location"),
        F.to_timestamp(F.col("date"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'").alias("reading_ts"),
        F.expr("try_cast(enter as int)").alias("enter_count"),  # confirmed a COUNT, not an hour — assumptions doc
        F.expr("try_cast(exit as int)").alias("exit_count"),
        F.col("id").alias("raw_id"),  # kept but never used as a key
        F.col("load_dt").alias("bronze_load_dt"),
        F.col("source").alias("bronze_source"),
    )
    .withColumn("silver_load_dt", F.current_timestamp()))


_CARS_VALID_FILTER = F.col("location").isNotNull() & F.col("reading_ts").isNotNull()


## cars_silver Table

In [0]:
@dlt.table(
    name="cars_silver",
    comment="Silver Streaming — traffic sensor counts. Deduplicated by (location, reading_ts) "
            "— UNVERIFIED grain (see requirements_and_assumptions.md); confirm via test_silver_day4.",
    table_properties=SILVER_PROPS,
)
@dlt.expect("valid_location", "location IS NOT NULL")
@dlt.expect("valid_reading_ts", "reading_ts IS NOT NULL")
def cars_silver():
    df = _transform_cars(_bronze_stream("cars_dlt"))
    df = deduplicate(df, ["location", "reading_ts"])
    return df.filter(_CARS_VALID_FILTER)

Cars Silver Quarantine

In [0]:
@dlt.table(
    name="cars_silver_quarantine",
    comment="Quarantine Streaming — car sensor rows with null/malformed location or date.",
    table_properties=QUARANTINE_PROPS,
)
def cars_silver_quarantine():
    df = _transform_cars(_bronze_stream("cars_dlt"))
    df = deduplicate(df, ["location", "reading_ts"])
    return (df.filter(~_CARS_VALID_FILTER)
            .withColumn("quarantine_reason",
                        F.when(F.col("location").isNull(), "MISSING_OR_MALFORMED_LOCATION")
                        .otherwise("UNPARSABLE_DATE"))
            .withColumn("quarantine_dt", F.current_timestamp()))

## Telegram Silver

##Transform Telegram

In [0]:
def _transform_telegram(df):
    return (df.select(
        clean_text(F.col("message")).alias("message"),
        F.col("date").alias("message_date"),
        F.expr("try_cast(hour as int)").alias("message_hour"),
        F.col("load_dt").alias("bronze_load_dt"),
        F.col("source").alias("bronze_source"),
    )
    .withColumn("silver_load_dt", F.current_timestamp()))


_TELEGRAM_VALID_FILTER = F.col("message").isNotNull() & (F.length(F.col("message")) > 0)

## Silver Telegram Table

In [0]:
@dlt.table(
    name="telegram_silver",
    comment="Silver Streaming — citizen traffic reports, whitespace-cleaned. "
            "Deduplicated by (message, message_date, message_hour) — the only available grain, no id column.",
    table_properties=SILVER_PROPS,
)
@dlt.expect("has_message", "message IS NOT NULL AND length(message) > 0")
def telegram_silver():
    df = _transform_telegram(_bronze_stream("telegram_dlt"))
    df = deduplicate(df, ["message", "message_date", "message_hour"])
    return df.filter(_TELEGRAM_VALID_FILTER)

##Telegram Silver Quarantine

In [0]:
@dlt.table(
    name="telegram_silver_quarantine",
    comment="Quarantine Streaming — telegram rows with null or empty message.",
    table_properties=QUARANTINE_PROPS,
)
def telegram_silver_quarantine():
    df = _transform_telegram(_bronze_stream("telegram_dlt"))
    df = deduplicate(df, ["message", "message_date", "message_hour"])
    return (df.filter(~_TELEGRAM_VALID_FILTER)
            .withColumn("quarantine_reason", F.lit("MISSING_OR_EMPTY_MESSAGE"))
            .withColumn("quarantine_dt", F.current_timestamp()))

## Node Locations Silver

## Transform Node Locations

In [0]:
def _transform_node_locations(df):
    return (df.select(
        F.expr("try_cast(location as int)").alias("location"),
        F.expr("try_cast(latitude as double)").alias("latitude"),
        F.expr("try_cast(longitude as double)").alias("longitude"),
        F.col("load_dt").alias("bronze_load_dt"),
        F.col("source").alias("bronze_source"),
    )
    .withColumn("silver_load_dt", F.current_timestamp()))


_NODE_VALID_FILTER = (
    F.col("location").isNotNull() &
    F.col("latitude").isNotNull() & F.col("longitude").isNotNull() &
    ~((F.col("latitude") == 0.0) & (F.col("longitude") == 0.0))
)

## Node Locations Silver Table


In [0]:
@dlt.table(
    name="node_locations_silver",
    comment="Silver — sensor coordinates. location=7's known (0,0) coordinate is quarantined, not silently kept.",
    table_properties=SILVER_PROPS,
)
@dlt.expect("valid_location", "location IS NOT NULL")
@dlt.expect("nonzero_coords", "NOT (latitude = 0.0 AND longitude = 0.0)")
def node_locations_silver():
    df = _transform_node_locations(_bronze_stream("node_locations_dlt"))
    return df.filter(_NODE_VALID_FILTER)

## Node Locations Silver Table Quarantine

In [0]:
@dlt.table(
    name="node_locations_silver_quarantine",
    comment="Quarantine — sensor rows with null or (0,0) coordinates. Expect exactly 1 row: location=7.",
    table_properties=QUARANTINE_PROPS,
)
def node_locations_silver_quarantine():
    df = _transform_node_locations(_bronze_stream("node_locations_dlt"))
    return (df.filter(~_NODE_VALID_FILTER)
            .withColumn("quarantine_reason",
                        F.when(F.col("location").isNull(), "MISSING_LOCATION")
                        .otherwise("ZERO_COORDINATES"))
            .withColumn("quarantine_dt", F.current_timestamp()))

## Streets List Silver

##Transform Streets List

In [0]:
def _transform_streets_list(df):
    return (df.select(
        F.expr("try_cast(street_id as int)").alias("street_id"),
        clean_text(F.col("street")).alias("street_name"),
        F.expr("try_cast(long as double)").alias("street_length_m"),
        F.expr("try_cast(latitude as double)").alias("latitude"),
        F.expr("try_cast(longitude as double)").alias("longitude"),
        F.expr("try_cast(dangerous as double)").alias("danger_score"),
        F.col("load_dt").alias("bronze_load_dt"),
        F.col("source").alias("bronze_source"),
    )
    .withColumn("silver_load_dt", F.current_timestamp()))


_STREETS_LIST_VALID_FILTER = F.col("street_id").isNotNull() & F.col("street_name").isNotNull()


## Streets List Silver Table

In [0]:
@dlt.table(
    name="streets_list_silver",
    comment="Silver — 36-row street dimension. FK target for streets_silver.street_id. "
            "street_name whitespace-cleaned for reliable joins to telegram_silver.message text.",
    table_properties=SILVER_PROPS,
)
@dlt.expect("valid_street_id", "street_id IS NOT NULL")
@dlt.expect("valid_street_name", "street_name IS NOT NULL")
def streets_list_silver():
    df = _transform_streets_list(_bronze_stream("streets_list_dlt"))
    return df.filter(_STREETS_LIST_VALID_FILTER)

## Streets List Silver Table Quarantine

In [0]:
@dlt.table(
    name="streets_list_silver_quarantine",
    comment="Quarantine — street dimension rows with null street_id or street_name.",
    table_properties=QUARANTINE_PROPS,
)
def streets_list_silver_quarantine():
    df = _transform_streets_list(_bronze_stream("streets_list_dlt"))
    return (df.filter(~_STREETS_LIST_VALID_FILTER)
            .withColumn("quarantine_reason", F.lit("MISSING_STREET_ID_OR_NAME"))
            .withColumn("quarantine_dt", F.current_timestamp()))